# Initiate

##  Requirements 

In [1]:
import gymnasium as gym
import numpy as np
import copy
from copy import deepcopy
import random 
import pickle
import sklearn
from sklearn import preprocessing
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn.metrics import accuracy_score
import time
import logging
from datetime import datetime
from stable_baselines3 import DQN
import pickle
from math import ceil
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import torch
from pymoo.algorithms.moo.nsga2 import calc_crowding_distance
from stable_baselines3.common.policies import obs_as_tensor


## RL

In [ ]:

class StoreAndTerminateWrapper(gym.Wrapper):
    '''
    :param env: (gym.Env) Gym environment that will be wrapped
    :param max_steps: (int) Max number of steps per episode
    '''
    def __init__(self, env):
        super(StoreAndTerminateWrapper,self).__init__(env)
        self.max_steps = 200
        self.current_step = 0
        self.env=env
        self.mem = []
        self.TotalReward = 0.0
        self.first_state = 0
        self.first_obs = 0
        self.prev_obs = 0
        self.states_list = []
        self.info = {}

    def reset(self, *args, **kwargs):
        self.current_step = 0
        obs, info = self.env.reset(*args, **kwargs)
        self.TotalReward = 0.0
        self.first_obs = obs
        return obs,info

    def step(self, action):
        if self.current_step == 0:
            self.prev_obs = self.first_obs
            self.first_state = deepcopy(self.env)
            self.states_list.append(self.first_state)
        self.current_step += 1
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.TotalReward += reward
        self.mem.append(tuple((self.prev_obs,action)))
        self.prev_obs = obs
        if self.current_step >= self.max_steps:
          truncated = True
        if obs[0] <= -1.2:
          truncated = True
          reward = -201 - self.TotalReward
          self.TotalReward = -200
        if terminated or truncated:
          self.mem.append(tuple(('done',self.TotalReward)))
        self.info['mem'] = self.mem
        self.info['state'] = self.states_list
        return obs, reward, terminated, truncated, info

    def set_state(self, state):
        self.env = deepcopy(state)
        obs = np.array(list(self.env.unwrapped.state))
        self.current_step = 0
        self.TotalReward = 0.0
        self.first_obs = obs
        return obs

def proportional_sampling_whitout_replacement(index , size):
  s=0
  s = sum(np.array(index))
  p = [ind/s for ind in index]
  samples = np.random.choice(index,size=size,replace=False,p=p)
  return samples


def population_sample(episodes , ind,  pop_size , threshold, functional_fault_size, reward_fault_size):
  """
  This function is meant to sample episodes from training after that you need to add test episodes using random_test 
  Set the parameters as you want but be careful the input episodes for this function is the memory of the agent and each step has seperate index 
  this function returns the final steps of the selected function then you need to extract that episodes from the input memore that is called 'episodes'
  use the episodes extract function ... 

  samples n episodes from training n1 functinal faults and n2 reward faults 
  reward faults are episodes with reward bellow the thresthreshold 
  from random test samples M episodes m1 random episode and
  m2 episodes with sudden reward change we dont have a sudden reward change in this example  
  """
  epsilon = 0.1
  index = []
  functional_fault = []
  reward_fault = []
  start_states =[]
  ind  = np.where(np.array(episodes)==('done',))
  index= ind[0]
  print(len(ind[0]),'episodes from training')
  population=[]
  for i in index:
    _,r = episodes[i]
    if abs(episodes[i-1][0][0])<(mtc_wrapped.low[0]+epsilon):
      functional_fault.append(i)
      print('function fault') 
    if r<threshold:
      reward_fault.append(i)
      print('reward fault')
  if len(functional_fault)<functional_fault_size:
    print('functional faults size is' ,len(functional_fault),' and its less than desired number' )
    population += functional_fault
    print('sampling more random episodes instead ...!')
  if len(functional_fault)==functional_fault_size:
    population += functional_fault
  if len(functional_fault)>functional_fault_size:
    # proportianl_sample_whitout_replacement()
    sam1=proportional_sampling_whitout_replacement(functional_fault,functional_fault_size)
    print(population)
    print("ff",len(functional_fault))
    population += sam1
  if len(reward_fault)<reward_fault_size:
    print('reward faults size is' ,len(reward_fault),' and its less than desired number' )
    population += reward_fault
    print('sampling more random episodes instead ...!')
  if len(reward_fault)==reward_fault_size:
    population += reward_fault
  if len(reward_fault)>reward_fault_size:
    #proportional sampling
    sam2 = proportional_sampling_whitout_replacement(reward_fault,reward_fault_size)
    population += list(sam2)
  r_size= pop_size-len(population)
  # random_test(model,env,r_size)
  print("RF",len(reward_fault))
  # population += reward_fault
  return population , r_size

def episode_extract(sampled_index, episodes):
  epis = []
  for i in sampled_index:
    # print(episodes[i])
    j = i-1
    while not episodes[j][0] == 'done':
      # print(episodes[j])
      if j==0:
        break
      j-=1
    slice1 = episodes[(j+1):(i+1)]
    epis.append(slice1)
    assert len(slice1)>0, 'Attempt to return Empty episode'
  return epis


def fitness_reward(episode):
  """
  here the reward could be calculated as the lengh of the episode; Since the
  reward of the cartpole is defined based on the number of steps without falling
  last part of the episode contains the signal of ('done',reward)
  """
  return len(episode)-1

def action_probability(model, state1, temperature=1.0):
  state_tensor = torch.as_tensor(state1).unsqueeze(0).to(model.device)
  with torch.no_grad(): 
    q_values = model.q_net(state_tensor).cpu().numpy()[0]
    scaled_q = q_values / temperature
    exp_q = np.exp(scaled_q)
    if exp_q.ndim == 2:
        # 批量处理情况
        return exp_q / np.sum(exp_q, axis=1, keepdims=True)
    else:
        # 单个观测情况
        return exp_q / np.sum(exp_q)


def fitness_confidence_general(episode, model, mode):
  """
  confidence level is define as differences between the highest and
  second highest action probabilities of selecting actions OR
  the ratio between the highest and lowest/second highest action probability
  :param `mode`: r for ration and m for differences 
  :param `model`: is the RL agent 
  :param `episode`: is the episode values or sequence from the rl 
  """
  cl = 0.0
  for i in range(len(episode)):
    if i==(len(episode)-1):
        if episode[i][0]=='done':
            return (cl/(len(episode)-1))
        else:
            assert False, "last state is not done , reward"
    else:
      prob=action_probability(model,episode[i][0])
      high1=prob.argmax()
      first = prob[high1]
      temp = prob
      temp[high1] = 0.0
      high2= temp.argmax()
      second = prob[high2]
      if mode == 'r':
        cl +=  (first/second)
        #In the next version this will be updated to a normalized ratio to avoid having large values 
      if mode == 'm':
        cl += (first - second) #To_Do: first - second / first +second this one is better 
  print("WARNING nothing returned", episode )


def fitness_reward_probability(ml, binary_episode):
  """
  This function returns the third fitness funciton that is ment to guide the search toward
  the episodes with a higher probability of a reward fault and as we have a minimizing 
  optimization funciton in MOSA we neeed to change this functionwe can either go with the
  negation of the probability of the reward fault = 1-probability of the reward fault
  that is equal to the probability of the bein a non-faulty episode
  :param `ml`: RF_FF_1rep for functional fault
  :param `binary episode`: episodes decodeed as having abstract states
  """
  # return -(ml.predict_proba(episode)[0][1])
  return ml.predict_proba(binary_episode)[0][0]

def fitness_functional_probability(ml, binary_episode):
  return ml.predict_proba(binary_episode)[0][0]

def abstract_state_general(model,state1,d):
  if type(state1) == str:
    if state1 == 'done':
      return 'end'
  state_tensor = torch.as_tensor(state1).unsqueeze(0).to(model.device)
    
  with torch.no_grad():  # 关闭梯度计算以提高效率
    q_values = model.q_net(state_tensor).cpu().numpy()[0]  # 获取第一个样本的Q值
    
  return tuple([ceil(q_value/d) for q_value in q_values])


#report function to check the performance metrics of the model
def report(model2,x_train, y_train,x_test, y_test):
  print("********************** reporting the result of the model **************************")
  print('The score for train data is {0}'.format(model2.score(x_train,y_train)))
  print('The score for test data is {0}'.format(model2.score(x_test,y_test)))


  predictions_train = model2.predict(x_train)
  predictions_test = model2.predict(x_test)

  print("\n\n--------------------------------------recall---------------------------------")

  print('the test recall for the class yes is {0}'.format(metrics.recall_score(y_test,predictions_test, pos_label=1)))
  print('the test recall for the class no is {0}'.format(metrics.recall_score(y_test,predictions_test, pos_label=0)))

  print('the training recall for the class yes is {0}'.format(metrics.recall_score(y_train,predictions_train, pos_label=1)))
  print('the training recall for the class no is {0}'.format(metrics.recall_score(y_train,predictions_train, pos_label=0)))


  print("\n\n--------------------------------------precision------------------------------")


  print('the test precision for the class yes is {0}'.format(metrics.precision_score(y_test,predictions_test, pos_label=1)))
  print('the test precision for the class no is {0}'.format(metrics.precision_score(y_test,predictions_test, pos_label=0)))

  print('the training precision for the class yes is {0}'.format(metrics.precision_score(y_train,predictions_train, pos_label=1)))
  print('the training precision for the class no is {0}'.format(metrics.precision_score(y_train,predictions_train, pos_label=0)))

  print("\n\n")
  print(classification_report(y_test, predictions_test, target_names=['NO ','yes']))

  tn, fp, fn, tp = confusion_matrix(y_test, predictions_test).ravel()
  specificity = tn / (tn+fp)
  print("\n\nspecifity :",specificity)
  print("\n\n--------------------------------------confusion----------------------------")
  CM = metrics.confusion_matrix(y_test, predictions_test)
  print("The confusion Matrix:")
  print(CM)
  print('the accuracy score in {0}\n\n'.format(accuracy_score(y_test, predictions_test)))
  print("********************** plotting the confusion matrix & ROC curve **************************")
  ConfusionMatrixDisplay(model2, x_test, y_test)
  metrics.plot_roc_curve(model2, x_test, y_test) 
  plt.show()

#dump

def dump_p(what, name):
  with open(f'/content/drive/MyDrive/MC/{name}.pickle', 'wb') as file:
      pickle.dump(what, file)


# write function for load

def load_p(name):
  with open(f'/content/drive/MyDrive/MC/{name}.pickle', 'rb') as file2:
    to_what = pickle.load(file2)
  return to_what
def local_load_p(name):
  with open(f'{name}', 'rb') as file2:
    to_what = pickle.load(file2)
  return to_what

def fix_testing(testing_episodes,testing_states,Env2):
  buffer =[] 
  episodes_set = []
  j=0
  for i in range(len(testing_episodes)):
    # Handle both array and scalar cases for the 'done' check
    if isinstance(testing_episodes[i][0], np.ndarray):
        # If it's an array, check if any element is 'done'
        is_done = (testing_episodes[i][0] == 'done').any()
    else:
        # If it's a scalar, do direct comparison
        is_done = testing_episodes[i][0] == 'done'
    
    if is_done:
        if i == 0:
            continue
        buffer.append(testing_episodes[i])
        episodes_set.append(buffer)
        buffer = []
    else:
        buffer.append(testing_episodes[i])
      # np.array(mtc_wrapped.set_state(qq[0]),dtype="float32")
  if not (episodes_set[0][0][0]==np.array(Env2.set_state(testing_states[0]),dtype="float32")).all():
    del testing_states[0]
  if not (episodes_set[0][0][0]==np.array(Env2.set_state(testing_states[0]),dtype="float32")).all():
    assert False, 'problem in starting states'
  if len(episodes_set)!=len(testing_states):
    del testing_states[-1]
  if len(episodes_set)!=len(testing_states):
    assert False, 'problem in data prepration'
  return episodes_set , testing_states


## ML

In [3]:
def Abstract_classes(ep,abstraction_d,model):
  d=abstraction_d
  abs_states1=[]
  for episode in ep:
    for state,action in episode:
      abs_st = abstract_state_general(model,state,d)
      if abs_st == 'end':
        continue
      abs_states1.append(abs_st)
  unique1=list(set(abs_states1))
  uni1 = np.array(unique1)
  a=len(abs_states1)
  b=len(set(abs_states1))
  print("abstract states:",b)
  print("Concrete states",a)
  print("ratio",b/a)
  return unique1,uni1


def ML_first_representation(Abs_d,epsilon_functional_fault_boarder,Reward_fault_boarder,uni1,model,ep,unique1):
  """
  TO-DO : fix epsilon and threshold
  """
  d = Abs_d
  # epsilon = 0.05
  epsilon = epsilon_functional_fault_boarder
  data1_x_b=[]
  data1_y_b= [] 
  data1_y_f_b = []
  functional_fault = False
  reward_fault_threshold =  Reward_fault_boarder

  for episode in ep:
    record = np.zeros(len(uni1))
    for state, action in episode:
      ab = abstract_state(model,state,d)
      if ab == 'end':
        print(action)
        if functional_fault:
          data1_y_f_b.append(1)
        else:
          data1_y_f_b.append(0)
        if action >= reward_fault_threshold:
          data1_y_b.append(0)
        else:
          data1_y_b.append(1)
        functional_fault=False
        continue
      if state[0] < (-1.2+epsilon) :
        # print("ff found")
        functional_fault = True
        print(state[0])
      ind = unique1.index(ab)
      # if len(w[0])>1:
        # print('error len is greater than 1')
      record[ind] = 1
      # if you want the frequency go with the next line 
      # record[ind] += 1
    data1_x_b.append(record)

  return data1_x_b, data1_y_b, data1_y_f_b

def ML_first_representation_func_based(Abs_d,functional_func,reward_func,model,input_episodes,unique1):
  """
  TO-DO : fix epsilon and threshold
  """
  d = Abs_d
  data1_x_b=[]
  data1_y_b= [] 
  data1_y_f_b = []
  for i, episode in enumerate(input_episodes):
    record = np.zeros(len(unique1))
    temp_flag = False
    for state, action in episode:
      ab = abstract_state_general(model,state,d)
      if ab == 'end':
        assert not temp_flag, f'Episode data problem, two terminations in one episode. Episode number{i}'
        temp_flag = True
        # print(action)
        # print(functional_func(episode))
        if functional_func(episode):
          data1_y_f_b.append(1)
        else:
          data1_y_f_b.append(0)
        if reward_func(episode):
          data1_y_b.append(1)
        else:
          data1_y_b.append(0)
        # print("end\n\n\n")
        # print(len(data1_y_b),"len(input_episodes)",len(input_episodes))
        continue
        # print(state[0])
      ind = unique1.index(ab)
      record[ind] = 1
      # print(state, action)
      assert len(data1_y_b)<len(input_episodes), "assert"
      # if you want the frequency go with the next line 
      # record[ind] += 1
    data1_x_b.append(record)

  return data1_x_b, data1_y_b, data1_y_f_b

## Genetic

In [ ]:
def translator(episode,model, d, unique5):
  """
  thid function takes the concrete episodes and returns the encoded episodes 
  based on the presence and absence of the individuals  
  :param 'episode': input episode
  :param 'model': RL model
  :param 'd': abstraction level = 1
  :param 'unique5': abstract classes 
  :return: encoded episodse based on the presence and absence

  """
  d=d
  record = np.zeros(len(unique5))
  for state, action in episode:
    ab = abstract_state_general(model,state,d)
    if ab == 'end':
      continue
    if ab in unique5:
      ind = unique5.index(ab)
    record[ind] = 1
  return [record]

def transform(state):
  position = state[0]
  noise = np.random.uniform(low=0.95, high=1.05)
  new_position= position * noise 
  new_state =deepcopy(state)
  new_state[0] = new_position 
  return new_state


def mutation_improved(population,model,env,objective_uncovered):
  """
  This is the final mutation function 
  It takes the population as input and returns the mutated individual
  :param 'population': Population that we want to mutate 
  :param 'model': RL model
  :param 'env': RL environment
  :param 'objective_uncovered: uncovered ubjectives for tournament selection
  :return: mutated candidate (we re-rexecute the episode from the mutation part)
  To-do:
  move deepcopy to the cadidate class methods .set info 
  """
  parent = tournament_selection(population, 10, objective_uncovered)  # tournament selection
  parent1 = deepcopy(parent.get_candidate_values())
  if len(parent1) < 3:
     assert False , "parent in mutation is shorter than 3"
  Mutpoint = random.randint(3,(len(parent1)-3))
  new_state = transform(parent1[Mutpoint][0])
  action = model.predict(new_state)
  if action[0]!= int(parent1[Mutpoint][1]):
    print('Mutation lured the agent ... ')
  new_parent = parent1[:Mutpoint]
  new_parent.append([new_state,'Mut'])
  new_cand =Candidate(new_parent)
  new_cand.set_start_state(parent.get_start_state())

  re_executed_epis = re_execute(model,env,new_cand)
  
  re_executed_cand = Candidate(re_executed_epis)
  re_executed_cand.set_start_state(new_cand.get_start_state())
  re_executed_cand.set_info(deepcopy(parent.get_info()))
  re_executed_cand.set_info(["mutation is done! ", "mutpoint was:",Mutpoint])

  
  return re_executed_cand

def mutation_improved_p(parent, model, env, m_rate):
    """
    这是最终的突变函数，输入考虑内部m_rate的父代
    根据给定的突变率m_rate，我们可能对 episodes 进行突变。
    :param 'parent' : 我们想要突变的个体
    :param 'model': RL模型
    :param 'env': RL环境
    :param 'm_rate': 突变率：推荐值为1/len(parent)
    :return : 突变后的个体
    To-do:
    将deepcopy移至候选.set信息
    """
    global MUTATION_NUMBER
    chance = random.uniform(0, 1)
    
    # 如果随机数大于突变率，则不进行突变，直接返回父代
    if chance > m_rate:
        return parent
    
    # 进行突变操作
    parent1 = deepcopy(parent.get_candidate_values())
    
    # 检查父代长度是否足够进行突变
    # 至少需要7个元素才能保证3到len-3之间有有效范围
    if len(parent1) < 7:
        # 长度不足，无法进行有效突变，返回原始父代
        return parent
    
    # 计算有效的突变点范围
    min_mut_point = 3
    max_mut_point = len(parent1) - 3
    
    # 再次确保范围有效（防御性检查）
    if min_mut_point > max_mut_point:
        return parent
    
    # 生成随机突变点
    Mutpoint = random.randint(min_mut_point, max_mut_point)
    
    # 执行突变操作
    new_state = transform(parent1[Mutpoint][0])
    action = model.predict(new_state, deterministic=True)
    
    if action[0] != int(parent1[Mutpoint][1]):
        print('Mutation lured the agent ... ')
    
    # 构建新的候选者
    new_parent = parent1[:Mutpoint]
    new_parent.append([new_state, 'Mut'])
    new_cand = Candidate(new_parent)
    new_cand.set_start_state(parent.get_start_state())
    
    # 重新执行并获取奖励
    re_executed_epis = re_execute(model, env, new_cand)
    n_reward = find_reward(re_executed_epis)
    re_executed_epis[-1] = ('done', n_reward)
    
    # 构建并返回重新执行后的候选者
    re_executed_cand = Candidate(re_executed_epis)
    re_executed_cand.set_start_state(new_cand.get_start_state())
    
    MUTATION_NUMBER += 1
    return re_executed_cand





def Crossover_improved_v2(population,model,d,objective_uncovered):
  """
  This is the crossover function that we are using 
  It takes the population as input and returns the mutated individual
  :param 'population': Population. we select a parent based on the tournament
   selection and then select the mutation point and then search for the matching point. 
  :param 'model': RL model
  :param 'env': RL environment
  :param 'objective_uncovered: uncovered ubjectives for tournament selection
  :return: mutated candidate (we re-rexecute the episode from the mutation part)
  To-do:
  finding matching episode could be improved bu storing a mapping between concrete states and  
  """
  found_match = False 
  while not (found_match):
    parent = tournament_selection(population, 10, objective_uncovered)  # tournament selection
    parent1 = deepcopy(parent.get_candidate_values())
    parent1_start_point = deepcopy(parent.get_start_state())
    if len(parent1)<4:
      assert False, 'input of crossover is shorter than expected '
    matches_list = []
    crosspoint = random.randint(1,(len(parent1)-3))
    abs_class = list(abstract_state_general(model,parent1[crosspoint][0],d))
    for i in range(50):
      indx = random.randint(0, len(population) - 1)
      random_candidate = deepcopy(population[indx])
      random_cand_data = random_candidate.get_candidate_values()
      random_cand_start_point = random_candidate.get_start_state()
      for st_index in range(1,len(random_cand_data)-3):
        random_ab = list(abstract_state_general(model,random_cand_data[st_index][0],d))
        if random_ab == abs_class:
          matches_list.append(st_index)
          found_match = True
      if found_match:
        break 
  # print('Crossover. attemp',i)
  index_match_in_matchlist = random.randint(0, len(matches_list) - 1)
  matchpoint = matches_list[index_match_in_matchlist]
  match_candidate =  deepcopy(random_candidate)
  match = deepcopy(random_cand_data)
  match_start = deepcopy(random_cand_start_point)
  offspring1 = deepcopy(parent1[:crosspoint])
  offspring1 += deepcopy(match[matchpoint:])
  new_reward1  = find_reward(offspring1)
  offspring1[-1] = ('done',new_reward1)
  candid1 = Candidate(offspring1)
  candid1.set_start_state(parent1_start_point)
  offspring2 = deepcopy(match[:matchpoint])
  offspring2 += deepcopy(parent1[crosspoint:])
  new_reward2  = find_reward(offspring2)
  offspring2[-1] = ('done',new_reward2)
  candid2 = Candidate(offspring2)
  candid2.set_start_state(match_start)
  if len(offspring1)<4:
    print(offspring1)
    assert False, 'created offspring 1 in crossover is shorter than expected '

  if len(offspring2)<4:
    print(offspring2)
    assert False, 'created offspring 2 in crossover is shorter than expected '

  return candid1, candid2

def find_reward(episode):
  if len(episode)>200:
    return -200
  if len(episode)<=200:
    if is_functional_fault_last_state(episode[-2],episode[-1]):
      return -200
    else:
      return -(len(episode)-1)

def Crossover_improved_v2_random(population,model,d,objective_uncovered):
  found_match = False 
  while not found_match:
    i = random.randint(0, len(population))
    parent1 = deepcopy(population[i].get_candidate_values())
    parent1_start_point = deepcopy(population[i].get_start_state())
    matches_list = []
    crosspoint = random.randint(1,(len(parent1)-3))
    abs_class = list(abstract_state(model,parent1[crosspoint][0],d))
    attemp = 0
    for i in range(700):
      attemp +=1
      indx = random.randint(0, len(population) - 1)
      random_candidate = deepcopy(population[indx])
      random_cand_data = random_candidate.get_candidate_values()
      random_cand_start_point = random_candidate.get_start_state()
      for st_index in range(1,len(random_cand_data)-3):
        random_ab = list(abstract_state(model,random_cand_data[st_index][0],d))
        if random_ab == abs_class:
          matches_list.append(st_index)
          found_match = True
      if found_match:
        break 
  print("match found in --- attemps",attemp)
  index_match_in_matchlist = random.randint(0, len(matches_list) - 1)
  matchpoint = matches_list[index_match_in_matchlist]
  match_candidate = random_candidate
  match = random_cand_data
  match_start = deepcopy(random_cand_start_point)
  offspring1 = deepcopy(parent1[:crosspoint])
  offspring1 += deepcopy(match[matchpoint:])
  offspring1[-1] = ['done',(len(offspring1)-1)]
  candid1 = Candidate(offspring1)
  candid1.set_start_state(parent1_start_point)

  offspring2 = deepcopy(match[:matchpoint])
  offspring2 += deepcopy(parent1[crosspoint:])
  offspring2[-1] = ['done',(len(offspring2)-1)]
  candid2 = Candidate(offspring2)
  candid2.set_start_state(match_start)
  return candid1, candid2

#updated for Mountain car
def re_execute(model,env,candidate):
  obs =env.reset()
  obs =env.set_state(deepcopy(candidate.get_start_state()))
  episode = candidate.get_candidate_values()
  steps_to_mut_point = len(episode)
  episode_reward = 0.0
  done= False 
  counter = 0 
  for i in range(steps_to_mut_point):
    action, _ = model.predict(obs, deterministic=True)
    action_selected = episode[i][1]
    if action_selected == 'Mut':
      # print(episode[i])
      # print(episode[i][0])
      action_selected, _ = model.predict(episode[i][0], deterministic=True)
      # print("ddd",i,"eee",steps_to_mut_point)
      # print(action_selected)
      # break
    obs, reward, terminated , truncated, info = env.step(int(action_selected)) # its very important to select the action here it means that we may 
    counter+=1
    #follow the previous path until the mutation point or we follow the route that the trained agent wants to follow forcing vs following 
    episode_reward += reward
    # print("counter",counter)
    if terminated or truncated:
      break 
  for j in range(200):
    if terminated or truncated:
      break
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated ,truncated, info = env.step(action) 
    counter+=1
    episode_reward += reward
  assert terminated or truncated
  if episode_reward>201:
    assert False 
  return env.info['mem'][-((counter)+1):]




In [5]:
#changed
import numpy as np
class Candidate:
    def __init__(self, candidates_vals):
        if isinstance(candidates_vals, (np.ndarray, np.generic)):
            self.candidate_values = candidates_vals.tolist()
        else:
            self.candidate_values = candidates_vals
        self.objective_values = []
        self.objectives_covered = []
        self.crowding_distance = 0
        self.uncertainity = []
        self.start_state = 0
        self.information = []
        self.mutation = False

    def get_candidate_values(self):
        return self.candidate_values

    def get_uncertainity_value(self, indx):
        return self.uncertainity[indx]
    def get_uncertainity_values(self):
        return self.uncertainity
    def set_uncertainity_values(self,uncertain):
        self.uncertainity = uncertain
    def set_candidate_values(self, cand):
        self.candidate_values = cand
    def set_candidate_values_at_index(self, indx,val):
        self.candidate_values[indx] = val

    def get_objective_values(self):
        return self.objective_values

    def get_objective_value(self, indx):
        return self.objective_values[indx]

    def set_objective_values(self, obj_vals):
        self.objective_values = obj_vals

    def add_objectives_covered(self, obj_covered):
        if obj_covered not in self.objectives_covered:
            self.objectives_covered.append(obj_covered)

    def get_covered_objectives(self):
        return self.objectives_covered

    def set_crowding_distance(self, cd):
        self.crowding_distance = cd

    def get_crowding_distance(self):
        return self.crowding_distance

    def exists_in_satisfied(self, indx):
        for ind in self.objectives_covered:
            if ind == indx:
                return True
        return False

    def is_objective_covered(self, obj_to_check):
        for obj in self.objectives_covered:
            if obj == obj_to_check:
                return True
        return False
    def set_start_state(self,start_point):
      self.start_state = deepcopy(start_point)

    def get_start_state(self):
      return self.start_state

    def set_info(self, new_information):
      self.information.append(new_information)
      
    def get_info(self):
      return self.information

    def mutated(self):
      self.mutation = True

In [6]:
def mutation_number_update(file_address,Mut_Num_to_add,iteration):
  if iteration == 0:
    with open(file_address, 'wb') as file:
      pickle.dump(Mut_Num_to_add, file)
    return
  with open(file_address, 'rb') as file2:
    Mut_num = pickle.load(file2)
  print(Mut_num)
  if type(Mut_num) == list:
    print('list')
    buffer = Mut_num
    buffer.append(Mut_Num_to_add)
    print(buffer)
  else:
    print('int')
    buffer =[] 
    buffer.append(Mut_num)
    buffer.append(Mut_Num_to_add)
    print(buffer)
  with open(file_address, 'wb') as file:
    pickle.dump(buffer, file)


## MOSA

In [7]:
scaler = preprocessing.StandardScaler()



# domination relation method, same as MOSA 
def dominates(value_from_pop, value_from_archive, objective_uncovered):
    dominates_f1 = False
    dominates_f2 = False
    for each_objective in objective_uncovered:
        f1 = value_from_pop[each_objective]
        f2 = value_from_archive[each_objective]
        if f1 < f2:
            dominates_f1 = True
        if f2 < f1:
            dominates_f2 = True
        if dominates_f1 and dominates_f2:
            break
    if dominates_f1 == dominates_f2:
        return False
    elif dominates_f1:
        return True
    return False




# calculating the fitness value function

def evaulate_population(func, pop , parameters):
    for candidate in pop:
      if isinstance(candidate, Candidate):
        # print(candidate.get_candidate_values())
        result = func(candidate.get_candidate_values())
        candidate.set_objective_values(result)
        print(candidate.get_objective_values())

def evaulate_population_with_archive(func, pop, already_executed):
    to_ret = []
    for candidate in pop:
        if isinstance(candidate, Candidate):
            if candidate.get_candidate_values() in already_executed:
                continue

            result = func(candidate.get_candidate_values())
            candidate.set_objective_values(result)
            already_executed.append(candidate.get_candidate_values())
            to_ret.append(candidate)
    return to_ret

def exists_in_archive(archive, index):
    for candidate in archive:
        if candidate.exists_in_satisfied(index):
            return True
    return False


# searching archive
def get_from_archive(obj_index, archive):
    for candIndx in range(len(archive)):
        candidate = archive[candIndx]
        if candidate.exists_in_satisfied(obj_index):
            return candidate, candIndx
    return None


# updating archive with adding the number of objective it satisfies, Same as Mosa paper
def update_archive(pop, objective_uncovered, archive, no_of_Objectives, threshold_criteria):
    for objective_index in range(no_of_Objectives):
        for pop_index in range(len(pop)):
            objective_values = pop[pop_index].get_objective_values()
            # if not objective_values[objective_index] or not threshold_criteria[objective_index]:
            if objective_values[objective_index] <= threshold_criteria[objective_index]:
                if exists_in_archive(archive, objective_index):
                    archive_value, cand_indx = get_from_archive(objective_index, archive)
                    obj_archive_values = archive_value.get_objective_values()
                    if obj_archive_values[objective_index] > objective_values[objective_index]:
                        value_to_add = pop[pop_index]
                        value_to_add.add_objectives_covered(objective_index)
                        # archive.append(value_to_add)
                        archive[cand_indx] = value_to_add
                        if objective_index in objective_uncovered:
                            objective_uncovered.remove(objective_index)
                        # archive.remove(archive_value)
                else:
                    value_to_add = pop[pop_index]
                    value_to_add.add_objectives_covered(objective_index)
                    archive.append(value_to_add)
                    if objective_index in objective_uncovered:
                        objective_uncovered.remove(objective_index)


# method to get the most dominating one
def select_best(tournament_candidates, objective_uncovered):
    best = tournament_candidates[0]  # in case none is dominating other
    for i in range(len(tournament_candidates)):
        candidate1 = tournament_candidates[i]
        for j in range(len(tournament_candidates)):
            candidate2 = tournament_candidates[j]
            if (dominates(candidate1.get_objective_values(), candidate2.get_objective_values(), objective_uncovered)):
                best = candidate1
    return best


def tournament_selection_improved(pop, size, objective_uncovered):
    tournament_candidates = []
    for i in range(size):
        indx = random.randint(0, len(pop) - 1)
        random_candidate = pop[indx]
        tournament_candidates.append(random_candidate)

    best = select_best(tournament_candidates, objective_uncovered)
    return best;


def tournament_selection(pop, size, objective_uncovered):
    tournament_candidates = []
    for i in range(size):
        indx = random.randint(0, len(pop) - 1)
        random_candidate = pop[indx]
        tournament_candidates.append(random_candidate)

    best = select_best(tournament_candidates, objective_uncovered)
    return best;




def generate_offspring_improved(population,model,env,d,objective_uncovered):
    population_to_return = []
    probability_C = 0.75
    probability_M = 0.3
    size = len(population)
    while (len(population_to_return) < size):
      probability_crossover = random.uniform(0, 1)
      if probability_crossover <= probability_C:  # 75% probability
        off1, off2 = Crossover_improved_v2(population,model,1,objective_uncovered)
        population_to_return.append(off1)
        population_to_return.append(off2)
      probability_mutation = random.uniform(0, 1)
      if probability_mutation <= probability_M:  # 30% probability this in for test purposes 
        off3 = mutation_improved(population, model,env,objective_uncovered)
        population_to_return.append(off3)
    return population_to_return






def generate_offspring_improved_v2(population,model,env,d,objective_uncovered):
    
    population_to_return = []
    probability_C = 0.75
    probability_M = 0.01
    size = len(population)
    while (len(population_to_return) < size):
      probability_crossover = random.uniform(0, 1)
      if probability_crossover <= probability_C:  # 75% probability
        parent1, parent2 = Crossover_improved_v2(population,model,d,objective_uncovered)
        parent1 = mutation_improved_p(parent1, model,env, (1 / len(parent1.get_candidate_values())))
        parent2 = mutation_improved_p(parent2, model,env, (1 / len(parent2.get_candidate_values())))
        population_to_return.append(parent1)
        population_to_return.append(parent2)

      if probability_crossover > probability_C:
        parent = tournament_selection(population, 10, objective_uncovered) #we may add a very small number of duplicated individulas but its not important as we are removing them in the final executions
        population_to_return.append(mutation_improved_p(parent, model,env,(1 / len(parent.get_candidate_values())))) 
      

    return population_to_return

def save_all_data(pop,no_of_Objectives,threshold_criteria, stored_data):
  '''
  This function will save all individulas with objective lower than treshhold 

  '''
  threshold_criteria_to_add_to_archive = [70, 0.06, 0.05, 0.05] 
  # be careful here ypu can set the satisfiing objectives that based on them you want to store the data  
  for individual in pop:
    individual_objective = individual.get_objective_values()
    for i in range(no_of_Objectives):
      if individual_objective[i]<threshold_criteria_to_add_to_archive[i]:
        # if individual not in stored_data:
        #   ind_ = deepcopy(individual)
        #   stored_data.append(ind_)
        # individual_objective_values = individual.get_objective_values()
        found = False
        for j in range(len(stored_data)):
          if individual_objective == stored_data[j].get_objective_values():
            found = True
            break
        if not found:
          ind_ = deepcopy(individual)
          stored_data.append(ind_)
  # return stored_data

def save_all_data2(pop, stored_data):
  '''
  This function will save all individulas in generations 
  you need to remove redundant data (based on fitness and ...)

  '''
  stored_data.append(list(pop))


def Build_Archive(pop,no_of_Objectives,threshold_criteria, stored_data, initial_population):
  '''
  If you are using the Archive of all generated episodes, this function
  removes the duplicated results and builds the Archive.
  :param 'pop': current generation
  :param 'no_of_Objectives': number of objectives
  :param 'threshold_criteria': threshold criteria (we are intrested in episodes that have fitness below these threshold values)
  :param 'stored_data': Archive of final episodes (return)
  :param 'initial_population': initial population. we are not considering these episodes in our archive for the second senario you need to add the number of faults, (implementation in RQ3)
  '''
  threshold_criteria_to_add_to_archive = threshold_criteria
# be careful as we can have different values for criterias here to add episodes to archive and for GA stopping criteria 
  for individual in pop:
    individual_objective = individual.get_objective_values()
    for i in range(no_of_Objectives):
      if individual_objective[i]<threshold_criteria_to_add_to_archive[i]:
        found = False
        for j in range(len(stored_data)):
          if individual_objective == stored_data[j].get_objective_values():
            found = True
            break
        for k in range(len(initial_population)):
          if individual_objective == initial_population[k].get_objective_values():
            found = True
            break
        if not found:
          ind_ = deepcopy(individual)
          stored_data.append(ind_)


### Sorting and RUN search

In [8]:

# finding best candidates and assigning to each front
def fast_dominating_sort(R_T, objective_uncovered):
    to_return = []
    front = []
    count = 0
    while len(R_T) > 1:
        count = 0
        for outer_loop in range(len(R_T)):
            best = R_T[outer_loop]
            add = True
            for inner_loop in range(len(R_T)):
                against = R_T[inner_loop]
                if best == against:
                    continue
                if (dominates(best.get_objective_values(), against.get_objective_values(), objective_uncovered)):
                    continue
                else:
                    add = False
                    break

            if add == True:
                if best not in front:
                    front.append(best)

                count = count + 1

        if len(front) > 0:
            to_return.append(front)
            for i in range(len(front)):
                R_T.remove(front[i])
                front = []

        if (len(to_return) == 0) or (count == 0):  # to check if no one dominates no one
            to_return.append(R_T)
            break

    return to_return


# sorting based on crowding distance
def sort_based_on_crowding_distance(e):
    values = e.get_crowding_distance()
    return values


def sort_based_on(e):
    values = e.get_objective_values()
    return values[0]


# sorting based on first objective value
def sort_worse(pop):
    pop.sort(key=sort_based_on, reverse=True)
    return pop
# preference sort, same as algorithm
def preference_sort(R_T, size, objective_uncovered):
    to_return = []
    for objective_index in objective_uncovered:
        min = 100
        best = R_T[0]
        for index in range(len(R_T)):
            objective_values = R_T[index].get_objective_values()
            if objective_values[objective_index] < min:
                min = objective_values[objective_index]
                best = R_T[index]
        to_return.append(best)
        R_T.remove(best)
    if len(R_T)>0:
        E = fast_dominating_sort(R_T, objective_uncovered)
        for i in range(len(E)):
            to_return.append(E[i])
    return to_return


# converting to numpy array (Required by library)
def get_array_for_crowding_distance(sorted_front):
    list = []
    for value in sorted_front:
        objective_values = value.get_objective_values()

        np_array = np.array(objective_values)
        list.append(np_array)

    np_list = np.array(list)
    cd = calc_crowding_distance(np_list)
    return cd
# method to assign each candidate its crownding distance

def assign_crowding_distance_to_each_value(sorted_front, crowding_distance):
    for candidate_index in range(len(sorted_front)):
        objective_values = sorted_front[candidate_index]
        objective_values.set_crowding_distance(crowding_distance[candidate_index])

def run_search(func, initial_population, no_of_Objectives, criteria,archive,logger,start,time_budget,size,d,env, parameters , second_archive,gens):
    global MUTATION_NUMBER
    MUTATION_NUMBER=0
    threshold_criteria = criteria 
    objective_uncovered = []
    print("initial population ",type(initial_population),len(initial_population))

    for obj in range(no_of_Objectives):
        objective_uncovered.append(obj)  # initializing number of uncovered objective

    random_population = initial_population 

    P_T = copy.copy(random_population)
    evaulate_population(func, random_population ,parameters)  # evaluating whole generation and storing results propabibly its with candidates

    # print(random_population[0].get_objective_values())
    update_archive(random_population, objective_uncovered, archive, no_of_Objectives,threshold_criteria)  # updating archive 
    # save initial population
    save_all_data2(random_population,gens)
    iteration = 0
    #limit of number of generations 
    while iteration <10:
        iteration = iteration + 1  # iteration count
        #To-DO: limit by the time budget instead of the generation number
        for arc in archive:
            logger.info("***ARCHIVE***")
            logger.info("\nValues: " + str(
                arc.get_candidate_values()) + "\nwith objective values: " + str(
                arc.get_objective_values()) + "\nSatisfying Objective: " + str(
                arc.get_covered_objectives()))
        print("Iteration count: " + str(iteration))
        logger.info("Iteration is : " + str(iteration))
        logger.info("Number of mutations : " + str(MUTATION_NUMBER))

        R_T = []
        
        Q_T = generate_offspring_improved_v2(P_T,model,env,d,objective_uncovered) #generate offsprings using crossover and mutation 

        evaulate_population(func, Q_T, parameters)  # evaluating offspring
        update_archive(Q_T, objective_uncovered, archive, no_of_Objectives, threshold_criteria)  # updating archive
        save_all_data(Q_T,no_of_Objectives,threshold_criteria,second_archive)
        # save generations
        save_all_data2(Q_T,gens)
        R_T = copy.deepcopy(P_T)  # R_T = P_T union Q_T
        R_T.extend(Q_T)

        F = preference_sort(R_T, size, objective_uncovered)  # Preference sorting and getting fronts

        if len(objective_uncovered) == 0:  # checking if all objectives are covered
            print("all_objectives_covered")
            logger.info("***Final-ARCHIVE***")
            print(("***Final-ARCHIVE***"))
            for arc in archive:
                print("\nValues: " + str(
                    arc.get_candidate_values()) + "\nwith objective values: " + str(
                    arc.get_objective_values()) + "\nSatisfying Objective: " + str(
                    arc.get_covered_objectives()))

                logger.info("\nValues: " + str(
                    arc.get_candidate_values()) + "\nwith objective values: " + str(
                    arc.get_objective_values()) + "\nSatisfying Objective: " + str(
                    arc.get_covered_objectives()))
            logger.info("Iteration is : "+str(iteration))
            logger.info("Number of mutations : "+str(MUTATION_NUMBER))
            break

        P_T_1 = []  # creating next generatint PT+1
        index = 0

        while len(P_T_1) <= size:  # if length of current generation is less that size of front at top then add it

            if not isinstance(F[index], Candidate):
                if len(P_T_1) + len(F[index]) > size:
                    break
            else:
                if len(P_T_1) + 1 > size:
                    break

            front = F[index]
            if isinstance(F[index], Candidate):  # if front contains only one item
                P_T_1.append(F[index])
                F.remove(F[index])
            else:
                for ind in range(len(F[index])):  # if front have multiple items
                    val = F[index][ind]
                    P_T_1.append(val)

                F.remove(F[index])
        while (len(P_T_1)) < size:  # crowding distance
            copyFront = copy.deepcopy(F[index])
            sorted_front = sort_worse(copyFront)  # sort before crowding distance

            crowding_distance = get_array_for_crowding_distance(sorted_front)  # coverting to libaray compaitble array
            assign_crowding_distance_to_each_value(sorted_front,
                                                   crowding_distance)  # assinging each solution its crowding distance
            sorted_front.sort(key=sort_based_on_crowding_distance, reverse=True)  # sorting based on crowding distance

            if (len(sorted_front) + len(
                    P_T_1)) > size:  # maintaining length and adding solutions with most crowding distances
                for sorted_front_indx in range(len(sorted_front)):
                    candidate = sorted_front[sorted_front_indx]
                    P_T_1.append(candidate)
                    if len(P_T_1) >= size:
                        break

            index = index + 1

        P_T_1 = P_T_1[0:size]
        P_T = P_T_1  # assigning PT+1 to PT


def minimize(func, population, lb, ub, no_of_Objectives, criteria,time_budget,logger,archive,size,d,env,parameters, second_archive,gens):
    assert hasattr(func, '__call__')

    start = time.time()
    run_search(func, population, no_of_Objectives, criteria,archive,logger,start,time_budget,size,d,env ,parameters, second_archive,gens)



In [9]:
class CartPole_caseStudy():
    def __init__(self):
        logger = logging.getLogger()

        now = datetime.now()
        log_file = 'output/STARLA' + str(i) + '_V2' + str(now) + '.log'
        logging.basicConfig(filename=log_file,
                            format='%(asctime)s %(message)s')
        self.parameters = [model,d,unique5]
        logger.setLevel(logging.WARNING)
    def _evaluate(self,x):
        fv = x
        model,d,unique5 = self.parameters
        obj1 = fitness_reward(fv)
        obj2 = fitness_confidence(fv,model,'m')
        binary_fv = translator(fv,model,d,unique5)
        obj3 = fitness_functional_probability(RF_FF_1rep,binary_fv)
        obj4 = fitness_functional_probability(RF_RF_1rep,binary_fv)
        to_ret = [obj1,obj2,obj3,obj4]
        logger = logging.getLogger()
        logger.info(str(fv)+","+str(to_ret))
        return to_ret


class MountainCar_caseStudy():
    def __init__(self):
        logger = logging.getLogger()
        now = datetime.now()
        log_file = 'log/STARLA' + str(i) + '_V2' + str(now) + '.log'
        logging.basicConfig(filename=log_file,
                            format='%(asctime)s %(message)s')
        self.parameters = [model,d,unique5]
        logger.setLevel(logging.WARNING)
    def _evaluate(self,x):
        fv = x
        model,d,unique5 = self.parameters
        obj1 = fitness_reward_general(fv)
        if obj1==None:
          debug_data1=[fv,x]
          with open(f'/content/drive/MyDrive/debug/data.pickle', 'wb') as file:
              pickle.dump(debug_data1, file)
          assert False
        obj2 = fitness_confidence_general(fv,model,'m')
        binary_fv = translator(fv,model,d,unique5)
        obj3 = fitness_functional_probability(RF_FF_1rep,binary_fv)
        # obj4 = fitness_functional_probability(RF_RF_1rep,binary_fv)
        to_ret = [obj1,obj2,obj3]
        logger = logging.getLogger()
        logger.info(str(fv)+","+str(to_ret))
        return to_ret


def run(i,population ,archive ,second_archive, gens):
    env=mtc_wrapped
    d=500
    size = len(population)
    lb = [0, 0, 0]
    ub = [100000,1000000,100000]

    parameters = [model,d,unique1]
    threshold_criteria = [-180, 0.04, 0.05]


    no_of_Objectives = 3;

    now = datetime.now()
    global logger
    logger = logging.getLogger()
    log_file = 'D:\\code\\RLtest\\starla\\data\\testcase\\Results' + str(i) + '_V2' + str(now).replace(":","_") + '.log'
    logging.basicConfig(filename=log_file,
                        format='%(asctime)s %(message)s')

    logger.setLevel(logging.WARNING)

    archive = minimize(MountainCar_caseStudy()._evaluate, population, lb, ub,
                       no_of_Objectives, threshold_criteria, 7200, 
                       logger,archive,size,d,env , parameters, second_archive,gens)
    logger.info("Iteration completed")
    logger.info("mu"+str(MUTATION_NUMBER))


### analyzer

In [10]:
def analyze_result(result):
  '''
  this function is to aggrigate the differences of the results 
  :param `result`: this is the output of the re-execution-improved function
  :return ``:
  '''
  total_dif =0
  # store_diff=[]
  for i in range(len(result)):
    dif = abs(result[i][1][0] - result[i][1][1])
    # store_diff.append([i,dif])
    total_dif += dif
  return total_dif #, store_diff


def get_objective_distribution_and_set_candidate_objectives(population,model,d,
                                                            unique1,RF_FF_1rep,
                                                            RF_RF_1rep):
  fit1_list =[]
  fit2_list =[]
  fit3_list =[]
  fit4_list =[]
  for i in range(len(population)):
    ind_data = population[i].get_candidate_values()
    fit1 = fitness_reward(ind_data)
    fit2 = fitness_confidence(ind_data,model,'m')
    binary_fv = translator(ind_data,model,d,unique1)
    fit3 = fitness_functional_probability(RF_FF_1rep,binary_fv)
    fit4 = fitness_reward_probability(RF_RF_1rep,binary_fv)
    obj = [fit1,fit2,fit3,fit4]
    population[i].set_objective_values(obj)
    fit1_list.append(fit1)
    fit2_list.append(fit2)
    fit3_list.append(fit3)
    fit4_list.append(fit4)
  return   fit1_list, fit2_list, fit3_list, fit4_list 

def get_3objective_distribution_and_set_candidate_objectives(population,model,d,
                                                            unique1,RF_FF_1rep):
  fit1_list =[]
  fit2_list =[]
  fit3_list =[]
  for i in range(len(population)):
    ind_data = population[i].get_candidate_values()
    fit1 = fitness_reward_general(ind_data)
    fit2 = fitness_confidence_general(ind_data,model,'m')
    binary_fv = translator(ind_data,model,d,unique1)
    fit3 = fitness_functional_probability(RF_FF_1rep,binary_fv)
    obj = [fit1,fit2,fit3]
    population[i].set_objective_values(obj)
    fit1_list.append(fit1)
    fit2_list.append(fit2)
    fit3_list.append(fit3)
  return   fit1_list, fit2_list, fit3_list 

def get_objective_distribution(population,model,d,unique1,RF_FF_1rep,RF_RF_1rep):
  fit1_list =[]
  fit2_list =[]
  fit3_list =[]
  fit4_list =[]
  for i in range(len(population)):
    ind_data = population[i].get_candidate_values()
    fit1 = fitness_reward(ind_data)
    fit2 = fitness_confidence(ind_data,model,'m')
    binary_fv = translator(ind_data,model,d,unique1)
    fit3 = fitness_functional_probability(RF_FF_1rep,binary_fv)
    fit4 = fitness_reward_probability(RF_RF_1rep,binary_fv)
    # obj = [fit1,fit2,fit3,fit4]
    # population[i].set_objective_values(obj)
    fit1_list.append(fit1)
    fit2_list.append(fit2)
    fit3_list.append(fit3)
    fit4_list.append(fit4)
  return   fit1_list, fit2_list, fit3_list, fit4_list 


def was_in_initial_population(solution, population,no_of_Objectives):
  flag = False
  for individuals_ in population:
    if individuals_.get_objective_values() == solution.get_objective_values():
      flag = True
  if not flag:
    return solution
  if flag:
    return 0

def analyze_set_differences(differences_set):
  '''
  input is a set of differences 
  '''
  analyzed_results=[]
  for item in differences_set:
    res = [len(item[0]),analyze_result(item[0]), item[1], len(item[0])/item[1]]
    analyzed_results.append(res)
  return analyzed_results

def extract_differences(solution_set):
  '''
  input is a set of solutions like archive or second_archive 
  the output a list ([list of differences as a result of re-execution],reward)
  '''
  differences = []
  for dastan in solution_set:
    reward = dastan.get_objective_values()[0]
    differences.append([re_execution_improved_v2(model,env,dastan),reward])
  return differences
  
def get_results_distribution(results):
  num_of_diff=[]
  diff_confi = []
  diff_ration = []
  for item in results:
    num_of_diff.append(item[0])
    diff_confi.append(item[1])
    diff_ration.append(item[3])
  return num_of_diff, diff_confi, diff_ration


# mountaincar

In [13]:
def random_test_2(model, env, Num):
    obs, info = env.reset()  # 解包获得观测值
    counter = 1
    episode_reward = 0.0

    for i in range(Num):
        # 只传递obs部分给predict
        action, _ = model.predict(obs, deterministic=True)
        # Gymnasium v0.29+的step()返回5个值（包括truncated）
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        episode_reward += reward
        if done:  # 检查episode结束
            counter += 1
            end = i
            episode_reward = 0.0
            obs, info = env.reset()  # 重置时再次解包

    iter = deepcopy(counter)
    u=1
    while iter > 1:
        if isinstance(env.info['mem'][-u][0], np.ndarray):
            if (env.info['mem'][-u][0] == 'done').any():
                lastpoint = -u
                iter -= 1
        else:
            if env.info['mem'][-u][0] == 'done':
                lastpoint = -u
                iter -= 1
        u += 1
    fin =Num - end
    start = -Num -counter
    randomtest = env.info['mem'][lastpoint:-fin]
    ran_state = env.info['state'][(-counter+1):-1]
    return randomtest , ran_state

In [14]:
def is_functional_fault(episode):
  epsilon = 0.1
  env = mtc_wrapped
  reward = episode[-1][1]
  last_state = episode[-2][0][0]
  if last_state<(env.low[0]+epsilon) and reward == -200:
    return True
  else:
    return False


def is_reward_fault(episode):
  RF_threshold = -180
  reward = episode[-1][1]
  # print(len(episode))
  if reward<RF_threshold and len(episode)>200:
    return True
  else:
    return False

def is_functional_fault_last_state(last_step,done_step):
  epsilon = 0.1
  env = mtc_wrapped
  assert done_step[0]=='done', "Wrong input!"
  reward = done_step[1]
  last_state = last_step[0][0]
  if last_state<(env.low[0]+epsilon) and reward == -200:
    return True
  else:
    return False


def is_reward_fault_last_state(last_step,done_step):
  RF_threshold = -199
  assert done_step[0]=='done', "Wrong input!"
  reward = done_step[1]
  last_state = last_step[0][0]
  # print(len(episode))
  if reward<RF_threshold and not is_functional_fault_last_state(last_step,done_step):
    return True
  else:
    return False


# Test

In [15]:
Drive_model  ="D:\\code\\RLtest\\1.zip"


mtc = gym.make('MountainCar-v0')
mtc_wrapped = StoreAndTerminateWrapper(mtc)
model = DQN('MlpPolicy',env=mtc_wrapped, verbose=1)
model = model.load(Drive_model)

RT,RTS = random_test_2(model,mtc_wrapped,200000)
FRT,FRTS = fix_testing(RT,RTS,mtc_wrapped)
RF=0
FF=0
Buff_reward = 0
Buff_len = 0
for test_episode in FRT:
    Buff_reward += test_episode[-1][1]
    Buff_len += (len(test_episode)-1)
    if is_functional_fault(test_episode):
        FF+=1
    if is_reward_fault(test_episode):
        RF+=1

def save_to_txt(FRT):
    
    with open(f'data\\testcase\\test-5.txt', 'w') as f:
        for i, episode in enumerate(FRT):
            f.write(f'Episode {i}:\n')
            for step in episode:
                # 将每个步骤的数据转换为字符串并写入
                f.write(f'{str(step)}\n')
            f.write('\n')  # 用空行分隔不同片段

save_to_txt(FRT)

Functional_Fault_rate = FF/len(FRT)
Reward_Fault_rate = RF/len(FRT)
print("Reward fault rate:",Reward_Fault_rate)
print("Functional fault rate:",Functional_Fault_rate)
print("average Reward:",Buff_reward/len(FRT))
print("average Lenght:",Buff_len/len(FRT))
print("Number of episodes:",len(FRT))
print("Number of functional faulty episodes:",FF)
print("Number of reward faulty episodes:",RF)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


d:\anaconda3\envs\mdpfuzz\lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.low to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.low` for environment variables or `env.get_wrapper_attr('low')` that will search the reminding wrappers.
  logger.warn(


Reward fault rate: 0.1609353507565337
Functional fault rate: 0.015130674002751032
average Reward: -138.2455295735901
average Lenght: 137.38308115543327
Number of episodes: 1454
Number of functional faulty episodes: 22
Number of reward faulty episodes: 234


# model and data

In [16]:
#Address of the trained RL model 
Drive_model  ="D:\\code\\RLtest\\1.zip"


mtc = gym.make('MountainCar-v0')
mtc_wrapped = StoreAndTerminateWrapper(mtc)
model = DQN('MlpPolicy',env=mtc_wrapped, verbose=1)
model = model.load(Drive_model)

#########################################################  Read DATA and Load Model #############


final_episodes = local_load_p(r"D:\edge\Dataset_STARLA\Dataset_MTC\Final_episodes_trainand_Test_2062_FIXED2.pickle")

######################################################### Read abstract classes #############


Read_from_data = True
d=500

if Read_from_data:
  with open(f"D:\\edge\\Dataset_STARLA\\Dataset_MTC\\Abstraction\\Abstraction_data_sampled_200_{d}.pickle", 'rb') as file2:
      unique1 = pickle.load(file2)
  uni1=np.array(unique1)
  unique5 = unique1
if not Read_from_data:
  unique1,uni1 = Abstract_classes(final_episodes,d,model)
  unique5 = unique1


epsilon = 0.1
data1_x_b, data1_y_b, data1_y_f_b = ML_first_representation_func_based(d,
                                                                       is_functional_fault,
                                                                       is_reward_fault
                                                                       ,model
                                                                       ,final_episodes
                                                                       ,unique1)

#########################################################  Train ML -  Reward fault predictor  #############

X_train_reward_fault, X_test_reward_fault, y_train_reward_fault, y_test_reward_fault = train_test_split(data1_x_b, data1_y_b, test_size=0.33, random_state=42)

RF_RF_1rep = RandomForestClassifier(random_state=0, class_weight='balanced')
RF_RF_1rep.fit(X_train_reward_fault,y_train_reward_fault)
#report(RF_RF_1rep,X_train_reward_fault,y_train_reward_fault,X_test_reward_fault,y_test_reward_fault)

#########################################################  Train ML - Functional fault predictor #############


X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(data1_x_b, data1_y_f_b, test_size=0.33, random_state=42)
RF_FF_1rep = RandomForestClassifier(random_state=0, class_weight='balanced')
RF_FF_1rep.fit(X_train_f,y_train_f)
#report(RF_FF_1rep,X_train_f,y_train_f,X_test_f,y_test_f)





Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## Abstractions and accuracy of ML

In [17]:
Drive_model  ="D:\\code\\RLtest\\1.zip"

mtc = gym.make('MountainCar-v0')
mtc_wrapped = StoreAndTerminateWrapper(mtc)
model = DQN('MlpPolicy',env=mtc_wrapped, verbose=1)
model = model.load(Drive_model)

final_episodes = local_load_p("D:\\edge\Dataset_STARLA\\Dataset_MTC\\Final_episodes_trainand_Test_2062_FIXED2.pickle")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [18]:
#generate and save abstract classes
d_set=[1]

report_functional = []
report_reward = []

for d in d_set:

  unique1,uni1 = Abstract_classes(final_episodes,d,model)
  with open(f"D:\\edge\\Dataset_STARLA\\Dataset_MTC\\Abstraction\\Abstraction_data_sampled_200_{d}.pickle", 'wb') as file:
    pickle.dump(unique1, file)

  
  data1_x_b, data1_y_b, data1_y_f_b = ML_first_representation_func_based(d,is_functional_fault,is_reward_fault,model,final_episodes,unique1)



  X_train_reward_fault, X_test_reward_fault, y_train_reward_fault, y_test_reward_fault = train_test_split(data1_x_b, data1_y_b, test_size=0.33, random_state=42)
  RF_RF_1rep = RandomForestClassifier(random_state=0, class_weight='balanced')
  RF_RF_1rep.fit(X_train_reward_fault,y_train_reward_fault)
  #report(RF_RF_1rep,X_train_reward_fault,y_train_reward_fault,X_test_reward_fault,y_test_reward_fault)
  #report_reward.append(classification_report(y_test_reward_fault, RF_RF_1rep.predict(X_test_reward_fault), target_names=['NO ','yes'],output_dict=True))
  
  #########################################################   ML #############
  print("####################################################  functional fault ####################\n\n\n")

  X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(data1_x_b, data1_y_f_b, test_size=0.33, random_state=42)
  RF_FF_1rep = RandomForestClassifier(random_state=0, class_weight='balanced')
  RF_FF_1rep.fit(X_train_f,y_train_f)
  #report(RF_FF_1rep,X_train_f,y_train_f,X_test_f,y_test_f)
  #report_functional.append(classification_report(y_test_f, RF_FF_1rep.predict(X_test_f), target_names=['NO ','yes'],output_dict=True))

  print("-------------------------------------------------------------------------------------------\n\n\n")

with open(f'D:\\code\\RLtest\\starla\\data\\testcase\\report_rf.pickle', 'wb') as file:
    pickle.dump(report_reward, file)
with open(f'D:\\code\\RLtest\\starla\\data\\testcase\\report_ff.pickle', 'wb') as file:
    pickle.dump(report_functional, file)

abstract states: 993
Concrete states 270740
ratio 0.0036677254930930045


d:\anaconda3\envs\mdpfuzz\lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.low to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.low` for environment variables or `env.get_wrapper_attr('low')` that will search the reminding wrappers.
  logger.warn(


####################################################  functional fault ####################



-------------------------------------------------------------------------------------------





# Execution 

In [19]:
#Address of the trained RL model 
Drive_model  ="D:\\code\\RLtest\\1.zip"

mtc = gym.make('MountainCar-v0')
mtc_wrapped = StoreAndTerminateWrapper(mtc)
model = DQN('MlpPolicy',env=mtc_wrapped, verbose=1)
model = model.load(Drive_model)

#########################################################  Read DATA and Load Model #############


final_episodes = local_load_p("D:\\edge\Dataset_STARLA\\Dataset_MTC\\Final_episodes_trainand_Test_2062_FIXED2.pickle")

######################################################### Read abstract classes #############
# data of abstract classes

Read_from_data = True
d=500

if Read_from_data:
  with open(f"D:\\edge\\Dataset_STARLA\\Dataset_MTC\\Abstraction\\Abstraction_data_sampled_200_{d}.pickle", 'rb') as file2:
      unique1 = pickle.load(file2)
  uni1=np.array(unique1)
  unique5 = unique1
if not Read_from_data:
  unique1,uni1 = Abstract_classes(final_episodes,d,model)
  unique5 = unique1


epsilon = 0.1
data1_x_b, data1_y_b, data1_y_f_b = ML_first_representation_func_based(d,
                                                                       is_functional_fault,
                                                                       is_reward_fault
                                                                       ,model
                                                                       ,final_episodes
                                                                       ,unique1)

#########################################################  Train ML -  Reward fault predictor  #############

X_train_reward_fault, X_test_reward_fault, y_train_reward_fault, y_test_reward_fault = train_test_split(data1_x_b, data1_y_b, test_size=0.33, random_state=42)

RF_RF_1rep = RandomForestClassifier(random_state=0, class_weight='balanced')
RF_RF_1rep.fit(X_train_reward_fault,y_train_reward_fault)
#report(RF_RF_1rep,X_train_reward_fault,y_train_reward_fault,X_test_reward_fault,y_test_reward_fault)

#########################################################  Train ML - Functional fault predictor #############


X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(data1_x_b, data1_y_f_b, test_size=0.33, random_state=42)
RF_FF_1rep = RandomForestClassifier(random_state=0, class_weight='balanced')
RF_FF_1rep.fit(X_train_f,y_train_f)
#report(RF_FF_1rep,X_train_f,y_train_f,X_test_f,y_test_f)





Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


d:\anaconda3\envs\mdpfuzz\lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.low to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.low` for environment variables or `env.get_wrapper_attr('low')` that will search the reminding wrappers.
  logger.warn(


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


# objectives

In [20]:
ee,qq=random_test_2(model,mtc_wrapped,400_000)  #create initial population with random test, in case if the environment is complex and this is time consuming you can use the episodes created for ml part 
test, teststate = fix_testing(ee,qq,mtc_wrapped)
print('len population',len(test))
start_state_ep1 = teststate
ep1=test
Sample_population = []
for i in range(0,1000):  #size of the initial population is 1500
    cd = Candidate(ep1[i])
    cd.set_start_state(start_state_ep1[i])
    Sample_population.append(cd)

len population 2868


# Run

In [21]:
MUTATION_NUMBER=0 #set the mutation counter to 0
#  from 7 i removed crossover logging 
Run_number = 1
for s in range(10):
  ee,qq=random_test_2(model,mtc_wrapped,400_000)  #create initial population with random test, in case of complex environments you can use the episodes created for ml part 
  test, teststate = fix_testing(ee,qq,mtc_wrapped)
  print('len population',len(test))
  start_state_ep1 = teststate
  ep1=test
  population = []
  for i in range(0,1500):  #size of the initial population is 1500
    cd = Candidate(ep1[i])
    cd.set_start_state(start_state_ep1[i])
    population.append(cd)
  archive1 = []
  second_arch1 =[]
  generations=[] # all of the episodes generated during the search
  run(0,population,archive1,second_arch1,generations)

  with open(f'D:\\code\\RLtest\\starla\\data\\testcase\\Results\\Dec07_arch1_population1000lastfull_run{Run_number}_{s}.pickle', 'wb') as file:
    pickle.dump(archive1, file)
  with open(f'D:\\code\\RLtest\\starla\\data\\testcase\\Results\\Dec07_second_arch1_population1000lastfull_run{Run_number}_{s}.pickle', 'wb') as file:
    pickle.dump(second_arch1, file)
  with open(f'D:\\code\\RLtest\\starla\\data\\testcase\\Results\\Dec07_generations_population1000lastfull_run{Run_number}_{s}.pickle', 'wb') as file:
    pickle.dump(generations, file)
  mutation_number_update(f'D:\\code\\RLtest\\starla\\data\\testcase\\Results\\Dec07Mutation_number_run{Run_number}.pickle',MUTATION_NUMBER,s)

len population 2868
initial population  <class 'list'> 1500
[-162.0, np.float32(0.14019334), np.float64(0.500336768781599)]
[-94.0, np.float32(0.5769648), np.float64(0.500336768781599)]
[-197.0, np.float32(0.1158668), np.float64(0.500336768781599)]
[-83.0, np.float32(0.4107716), np.float64(0.500336768781599)]
[-125.0, np.float32(0.43444332), np.float64(0.500336768781599)]
[-97.0, np.float32(0.55513877), np.float64(0.500336768781599)]
[-89.0, np.float32(0.56402177), np.float64(0.500336768781599)]
[-85.0, np.float32(0.4534906), np.float64(0.500336768781599)]
[-83.0, np.float32(0.42687392), np.float64(0.500336768781599)]
[-99.0, np.float32(0.53613424), np.float64(0.500336768781599)]
[-90.0, np.float32(0.5731503), np.float64(0.500336768781599)]
[-136.0, np.float32(0.39791888), np.float64(0.500336768781599)]
[-128.0, np.float32(0.43133244), np.float64(0.500336768781599)]
[-195.0, np.float32(0.11619892), np.float64(0.500336768781599)]
[-126.0, np.float32(0.43853635), np.float64(0.50033676878

d:\anaconda3\envs\mdpfuzz\lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.low to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.low` for environment variables or `env.get_wrapper_attr('low')` that will search the reminding wrappers.
  logger.warn(


[-147, np.float32(0.057066437), np.float64(0.500336768781599)]
[-146, np.float32(0.32539454), np.float64(0.500336768781599)]
[-145.0, np.float32(0.31097677), np.float64(0.500336768781599)]
[-185, np.float32(0.21083033), np.float64(0.500336768781599)]
[-61, np.float32(0.41866213), np.float64(0.500336768781599)]
[-95, np.float32(0.49819982), np.float64(0.500336768781599)]
[-75, np.float32(0.44581747), np.float64(0.500336768781599)]
[-131.0, np.float32(0.42154503), np.float64(0.500336768781599)]
[-89.0, np.float32(0.5722157), np.float64(0.500336768781599)]
[-194, np.float32(0.25962967), np.float64(0.500336768781599)]
[-50, np.float32(0.17506143), np.float64(0.500336768781599)]
[-200, np.float32(0.010878902), np.float64(0.500336768781599)]
[-62, np.float32(0.012610506), np.float64(0.500336768781599)]
[-158.0, np.float32(0.13800824), np.float64(0.500336768781599)]
[-89.0, np.float32(0.5644718), np.float64(0.500336768781599)]
[-194, np.float32(0.18008053), np.float64(0.500336768781599)]
[-51

d:\anaconda3\envs\mdpfuzz\lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.low to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.low` for environment variables or `env.get_wrapper_attr('low')` that will search the reminding wrappers.
  logger.warn(


Mutation lured the agent ... 
[-63, np.float32(0.23320577), np.float64(0.500336768781599)]
[-78, np.float32(0.4729489), np.float64(0.500336768781599)]
[-89, np.float32(0.61499834), np.float64(0.500336768781599)]
[-123, np.float32(0.40592584), np.float64(0.500336768781599)]
[-35, np.float32(0.4342664), np.float64(0.500336768781599)]
[-91, np.float32(0.22536246), np.float64(0.500336768781599)]
[-200, np.float32(0.0880905), np.float64(0.500336768781599)]
[-77, np.float32(0.47847128), np.float64(0.500336768781599)]
[-180, np.float32(0.034476638), np.float64(0.500336768781599)]
[-110, np.float32(0.4628726), np.float64(0.500336768781599)]
[-146, np.float32(0.038545705), np.float64(0.500336768781599)]
[-89, np.float32(0.34752905), np.float64(0.500336768781599)]
[-170, np.float32(0.2919372), np.float64(0.500336768781599)]
[-139, np.float32(0.33394584), np.float64(0.500336768781599)]
[-37, np.float32(0.45806065), np.float64(0.500336768781599)]
[-124, np.float32(0.22154875), np.float64(0.5003367

...

d:\anaconda3\envs\mdpfuzz\lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.low to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.low` for environment variables or `env.get_wrapper_attr('low')` that will search the reminding wrappers.
  logger.warn(


Mutation lured the agent ... 
[-200, np.float32(0.24387959), np.float64(0.500336768781599)]
[-126, np.float32(0.39660197), np.float64(0.500336768781599)]
[-17, np.float32(0.005908212), np.float64(0.500336768781599)]
[-36, np.float32(0.16048984), np.float64(0.500336768781599)]
[-200, np.float32(0.14260599), np.float64(0.500336768781599)]
[-200, np.float32(0.11319449), np.float64(0.500336768781599)]
[-78, np.float32(0.09954645), np.float64(0.500336768781599)]
[-37, np.float32(0.60897684), np.float64(0.500336768781599)]
[-200, np.float32(0.5758929), np.float64(0.500336768781599)]
[-58, np.float32(0.21780685), np.float64(0.500336768781599)]
[-76, np.float32(0.49741706), np.float64(0.500336768781599)]
[-164, np.float32(0.56744015), np.float64(0.500336768781599)]
[-200, np.float32(0.27098295), np.float64(0.500336768781599)]
[-138, np.float32(0.57931197), np.float64(0.500336768781599)]
[-30, np.float32(0.432054), np.float64(0.500336768781599)]
[-22, np.float32(0.15852629), np.float64(0.500336

d:\anaconda3\envs\mdpfuzz\lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.low to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.low` for environment variables or `env.get_wrapper_attr('low')` that will search the reminding wrappers.
  logger.warn(


[-41, np.float32(0.31704167), np.float64(0.500336768781599)]
[-200, np.float32(0.4631078), np.float64(0.500336768781599)]
[-109, np.float32(0.30529118), np.float64(0.500336768781599)]
[-111, np.float32(0.634976), np.float64(0.500336768781599)]
[-120, np.float32(0.012705463), np.float64(0.500336768781599)]
[-90, np.float32(0.13243616), np.float64(0.500336768781599)]
[-179, np.float32(0.48119086), np.float64(0.500336768781599)]
[-10, np.float32(0.2336537), np.float64(0.500336768781599)]
[-72, np.float32(0.5559385), np.float64(0.500336768781599)]
[-188, np.float32(0.16095406), np.float64(0.500336768781599)]
[-33, np.float32(0.35965225), np.float64(0.500336768781599)]
[-54, np.float32(0.4581683), np.float64(0.500336768781599)]
[-180, np.float32(0.33685908), np.float64(0.500336768781599)]
[-200, np.float32(0.57847553), np.float64(0.500336768781599)]
[-150, np.float32(0.22799744), np.float64(0.500336768781599)]
[-73, np.float32(0.52205086), np.float64(0.500336768781599)]
[-88, np.float32(0.1

KeyboardInterrupt: 